# Analysis

Reads only the cached decision tables in `results/tables` (produced by `python -m src.evaluate` from the cached raw model
outputs) and produces every figure and table in the paper, plus the macros the paper includes.

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/craigouma/llm-underwriting-injection"
REPO = Path(os.environ.get("UW_REPO_DIR", "/kaggle/working/repo"))
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print("repo", REPO)

In [ ]:
import pandas as pd
from src import analysis as an, figures, report
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 200)
RUN = os.environ.get("UW_RUN", "main")
df = an.with_oracle(an.load(RUN))
len(df)

## Clean utility and extraction accuracy (RQ3)

In [ ]:
display(an.clean_utility(df).round(3)); display(an.extraction_accuracy(df).pivot(index='field', columns='model', values='accuracy').round(3))

## Attack success rate by architecture and category (RQ1, RQ2)

In [ ]:
display(report.pooled_asr(df)[['model','arm','category','k','n','rate','lo','hi']].round(3))
display(an.clean_approve_rate(df).round(3))

## Planned paired tests, Holm-adjusted per model

In [ ]:
display(an.family_tests(df).round(4))

## Oracle attribution, review burden and plausibility trend (RQ2, RQ3, H2, H3)

In [ ]:
display(an.oracle_attribution(df).round(3)); display(an.review_burden(df).round(3)); display(an.caught_by_c(df).round(3)); display(an.plausibility_trend(df).round(3))

## Figures and paper tables

In [ ]:
from IPython.display import display
paths = figures.make_all(df)
report.main(RUN)
print([str(p) for p in paths])